In [ ]:
# Install necessary packages

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from codecarbon import EmissionsTracker
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import time
from tqdm import tqdm
import sys

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Start tracking carbon emissions for the entire process
tracker = EmissionsTracker()
tracker.start()

# Load the dataset
data = pd.read_csv('data/hackathon_dataset.csv')

# Display basic information
print(f"Dataset shape: {data.shape}")
print(f"Number of product descriptions: {len(data)}")


In [ ]:
# Identify text column and label columns
text_column = 'text_raw'
# All columns except 'text_raw' and 'row_num' are label columns
label_columns = [col for col in data.columns if col not in ['text_raw', 'row_num']]

# Check for missing values
print("Missing values in the dataset:")
print(data.isnull().sum())

# Analyze text length
data['text_length'] = data[text_column].apply(len)
print("\nText length statistics:")
print(data['text_length'].describe())

# Plot text length distribution
plt.figure(figsize=(10, 6))
sns.histplot(data['text_length'], bins=50, kde=True)
plt.title("Distribution of Product Description Lengths")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.show()

# Check label distribution
label_counts = data[label_columns].sum().sort_values(ascending=False)
print("\nLabel distribution (top 10):")
print(label_counts.head(10))

# Visualize label distribution
plt.figure(figsize=(12, 8))
label_counts.head(20).plot(kind='bar')
plt.title('Top 20 Skin Conditions/Attributes in the Dataset')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate correlation between labels
plt.figure(figsize=(16, 12))
corr_matrix = data[label_columns].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', annot=False,
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5)
plt.title('Correlation Between Labels')
plt.tight_layout()
plt.show()

# Check average number of labels per product
data['label_count'] = data[label_columns].sum(axis=1)
print("\nNumber of labels per product:")
print(data['label_count'].describe())
plt.figure(figsize=(10, 6))
sns.histplot(data['label_count'], bins=15, kde=True)
plt.title("Distribution of Number of Labels per Product")
plt.xlabel("Number of Labels")
plt.ylabel("Frequency")
plt.show()


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from codecarbon import EmissionsTracker

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Start tracking carbon emissions
preprocess_tracker = EmissionsTracker()
preprocess_tracker.start()

def preprocess_text(text):
    """Clean and preprocess text data"""
    # Handle NaN values
    if pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords and lemmatize
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    # Keep domain-specific stopwords that might be important for skin conditions
    skin_terms = {'oily', 'dry', 'sensitive', 'normal', 'combination', 'acne', 'spf', 'skin'}
    filtered_stop_words = stop_words - skin_terms

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in filtered_stop_words]

    return ' '.join(words)

# Apply preprocessing to the text data
print("Preprocessing text descriptions...")
data['processed_text'] = data[text_column].apply(preprocess_text)

# Display sample
print("\nSample of preprocessed text:")
for original, processed in zip(data[text_column].head(2), data['processed_text'].head(2)):
    print(f"Original: {original[:100]}...")
    print(f"Processed: {processed[:100]}...")
    print("-" * 50)

# Stop tracking and get emissions
emissions = preprocess_tracker.stop()
print(f"Carbon emissions for preprocessing: {emissions} kg CO2eq")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
import os

# Fix the codecarbon lock file issue
# Check if the lock file exists and remove it if it does
if os.path.exists('/tmp/.codecarbon.lock'):
    os.remove('/tmp/.codecarbon.lock')

# Start the tracker with allow_multiple_runs set to True
from codecarbon import EmissionsTracker
feature_tracker = EmissionsTracker(allow_multiple_runs=True)
feature_tracker.start()

# Create TF-IDF features with a limited number of features for a lightweight model
print("Creating TF-IDF features...")
tfidf_vectorizer = TfidfVectorizer(
    max_features=1500,  # Limit features for lightweight model
    ngram_range=(1, 2),  # Use unigrams and bigrams
    min_df=2,           # Minimum document frequency
    max_df=0.9,         # Maximum document frequency
    stop_words='english' # Additional stopwords removal
)

# Transform text data to TF-IDF features
X = tfidf_vectorizer.fit_transform(data['processed_text'])
y = data[label_columns].values

print(f"Initial feature matrix shape: {X.shape}")

# Feature selection to further reduce dimensionality
print("Applying feature selection...")
k = min(1000, X.shape[1])  # Select top features
selector = SelectKBest(chi2, k=k)
X_selected = selector.fit_transform(X, y.sum(axis=1))

print(f"Reduced feature matrix shape: {X_selected.shape}")

# Split into training and test sets without using stratify parameter
# This avoids the error with classes that have only 1 sample
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

# Stop tracking and capture emissions
emissions = feature_tracker.stop()
print(f"Carbon emissions for feature engineering: {emissions} kg CO2eq")



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.multioutput import MultiOutputClassifier
import joblib

def train_evaluate_model(model_name, base_model, X_train, y_train, X_test, y_test):
    """Train and evaluate a multi-label classification model"""
    # Start tracking carbon emissions
    model_tracker = EmissionsTracker()
    model_tracker.start()

    print(f"Training {model_name}...")

    # Create and train model
    multi_model = MultiOutputClassifier(base_model, n_jobs=-1)
    multi_model.fit(X_train, y_train)

    # Make predictions
    y_pred = multi_model.predict(X_test)

    # Calculate custom metrics for multi-label classification
    # Sample accuracy (exact matches)
    sample_accuracy = np.mean(np.all(y_test == y_pred, axis=1))

    # Calculate F1 scores for each label
    f1_scores = []
    for i in range(y_test.shape[1]):
        # Handle case where a column might have all 0s or all 1s
        if len(np.unique(y_test[:, i])) < 2 or len(np.unique(y_pred[:, i])) < 2:
            f1 = 1.0 if np.array_equal(y_test[:, i], y_pred[:, i]) else 0.0
        else:
            # Calculate F1 score with appropriate setting
            from sklearn.metrics import f1_score as skf1
            f1 = skf1(y_test[:, i], y_pred[:, i], average='binary')
        f1_scores.append(f1)

    # Average F1 score across all labels
    f1_macro = np.mean(f1_scores)

    # Custom hamming loss (proportion of incorrectly predicted labels)
    hamming_loss = 1 - np.mean(y_test == y_pred)

    # Overall accuracy (proportion of correctly predicted labels)
    overall_accuracy = np.mean(y_test == y_pred)

    # Estimate model size
    model_size = sys.getsizeof(multi_model) / (1024 * 1024)  # Size in MB

    # Stop tracking emissions
    model_tracker.stop()
    emissions = model_tracker.final_emissions

    return {
        'model_name': model_name,
        'model': multi_model,
        'sample_accuracy': sample_accuracy,
        'overall_accuracy': overall_accuracy,
        'f1_macro': f1_macro,
        'f1_scores': f1_scores,
        'hamming_loss': hamming_loss,
        'emissions': emissions,
        'model_size': model_size
    }

# Define models to try (lightweight options)
models = [
    ('LogisticRegression', LogisticRegression(
        C=1.0,
        solver='liblinear',
        max_iter=1000,
        class_weight='balanced'
    )),
    ('LinearSVC', LinearSVC(
        C=1.0,
        max_iter=2000,
        class_weight='balanced',
        dual=False
    )),
    ('RandomForest', RandomForestClassifier(
        n_estimators=50,  # Reduced for lightweight model
        max_depth=10,
        min_samples_split=5,
        class_weight='balanced'
    ))
]

# Train and evaluate each model
results = []
for name, model in models:
    result = train_evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results.append(result)

    print(f"\nResults for {name}:")
    print(f"Sample Accuracy (exact matches): {result['sample_accuracy']:.4f}")
    print(f"Overall Accuracy: {result['overall_accuracy']:.4f}")
    print(f"F1 Score (Macro): {result['f1_macro']:.4f}")
    print(f"Hamming Loss: {result['hamming_loss']:.4f}")
    print(f"Carbon Emissions: {result['emissions']:.6f} kg CO2eq")
    print(f"Model Size: {result['model_size']:.2f} MB")

    # Check if model meets accuracy threshold
    if result['overall_accuracy'] >= 0.85:
        print(f"✅ Model meets accuracy threshold (>= 85%)")
    else:
        print(f"❌ Model doesn't meet accuracy threshold (>= 85%)")

# Find the best model based on F1 score and model size
best_model = max(results, key=lambda x: x['f1_macro'])
print(f"\nBest model: {best_model['model_name']} with F1 Score: {best_model['f1_macro']:.4f}")


In [ ]:
# If the best model doesn't reach 85% accuracy, let's optimize it
from sklearn.model_selection import GridSearchCV

optimization_tracker = EmissionsTracker()
optimization_tracker.start()

if best_model['overall_accuracy'] < 0.85:
    print("\nOptimizing model to improve performance...")

    # Hyperparameter tuning for the best model
    if best_model['model_name'] == 'LogisticRegression':
        # Define parameter grid for LogisticRegression
        param_grid = {
            'estimator__C': [0.1, 0.5, 1.0, 5.0],
            'estimator__penalty': ['l1', 'l2']
        }
        base_model = LogisticRegression(
            solver='liblinear',
            max_iter=1000,
            class_weight='balanced'
        )

    elif best_model['model_name'] == 'LinearSVC':
        # Define parameter grid for LinearSVC
        param_grid = {
            'estimator__C': [0.1, 0.5, 1.0, 5.0],
            'estimator__loss': ['hinge', 'squared_hinge']
        }
        base_model = LinearSVC(
            max_iter=2000,
            class_weight='balanced',
            dual=False
        )

    else:  # RandomForest
        # Define parameter grid for RandomForest
        param_grid = {
            'estimator__n_estimators': [30, 50, 100],
            'estimator__max_depth': [8, 10, 15]
        }
        base_model = RandomForestClassifier(
            min_samples_split=5,
            class_weight='balanced'
        )

    # Set up GridSearchCV for multioutput classification
    # We'll optimize for one label to save computational resources
    print("Performing hyperparameter tuning...")
    multi_model = MultiOutputClassifier(base_model)

    # Use a subset of the training data to reduce computational cost
    X_sample = X_train[:1000]
    y_sample = y_train[:1000, 0].reshape(-1, 1)  # Use only the first label for tuning

    grid_search = GridSearchCV(
        multi_model,
        param_grid,
        cv=3,
        scoring='f1_macro',
        n_jobs=-1
    )

    grid_search.fit(X_sample, y_sample)

    print(f"Best parameters: {grid_search.best_params_}")

    # Use the best parameters to train the full model on all labels
    if best_model['model_name'] == 'LogisticRegression':
        optimized_base_model = LogisticRegression(
            C=grid_search.best_params_['estimator__C'],
            penalty=grid_search.best_params_['estimator__penalty'],
            solver='liblinear',
            max_iter=1000,
            class_weight='balanced'
        )

    elif best_model['model_name'] == 'LinearSVC':
        optimized_base_model = LinearSVC(
            C=grid_search.best_params_['estimator__C'],
            loss=grid_search.best_params_['estimator__loss'],
            max_iter=2000,
            class_weight='balanced',
            dual=False
        )

    else:  # RandomForest
        optimized_base_model = RandomForestClassifier(
            n_estimators=grid_search.best_params_['estimator__n_estimators'],
            max_depth=grid_search.best_params_['estimator__max_depth'],
            min_samples_split=5,
            class_weight='balanced'
        )

    # Train the optimized model on the full dataset
    print("Training optimized model...")
    optimized_model = MultiOutputClassifier(optimized_base_model, n_jobs=-1)
    optimized_model.fit(X_train, y_train)

    # Evaluate optimized model
    y_pred_opt = optimized_model.predict(X_test)

    # Calculate metrics
    sample_accuracy_opt = np.mean(np.all(y_test == y_pred_opt, axis=1))
    overall_accuracy_opt = np.mean(y_test == y_pred_opt)

    # Calculate F1 scores for each label
    f1_scores_opt = []
    for i in range(y_test.shape[1]):
        if len(np.unique(y_test[:, i])) < 2 or len(np.unique(y_pred_opt[:, i])) < 2:
            f1 = 1.0 if np.array_equal(y_test[:, i], y_pred_opt[:, i]) else 0.0
        else:
            from sklearn.metrics import f1_score as skf1
            f1 = skf1(y_test[:, i], y_pred_opt[:, i], average='binary')
        f1_scores_opt.append(f1)

    f1_macro_opt = np.mean(f1_scores_opt)
    hamming_loss_opt = 1 - overall_accuracy_opt

    # Calculate model size
    opt_model_size = sys.getsizeof(optimized_model) / (1024 * 1024)  # Size in MB

    print("\nOptimized Model Results:")
    print(f"Sample Accuracy (exact matches): {sample_accuracy_opt:.4f}")
    print(f"Overall Accuracy: {overall_accuracy_opt:.4f}")
    print(f"F1 Score (Macro): {f1_macro_opt:.4f}")
    print(f"Hamming Loss: {hamming_loss_opt:.4f}")
    print(f"Model Size: {opt_model_size:.2f} MB")

    # Update best model if optimized is better
    if f1_macro_opt > best_model['f1_macro']:
        print("Optimized model is better than the previous best model!")
        best_model = {
            'model_name': f"Optimized {best_model['model_name']}",
            'model': optimized_model,
            'sample_accuracy': sample_accuracy_opt,
            'overall_accuracy': overall_accuracy_opt,
            'f1_macro': f1_macro_opt,
            'f1_scores': f1_scores_opt,
            'hamming_loss': hamming_loss_opt,
            'model_size': opt_model_size
        }
else:
    print("Best model already meets accuracy threshold. Skipping optimization.")

optimization_tracker.stop()
opt_emissions = optimization_tracker.final_emissions
print(f"Carbon emissions for optimization: {opt_emissions:.6f} kg CO2eq")


In [ ]:
# Analyze feature importance for the best model
importance_tracker = EmissionsTracker()
importance_tracker.start()

print("\nAnalyzing feature importance...")

# Get feature names from TF-IDF vectorizer
if hasattr(selector, 'get_support'):
    # Get selected feature indices
    selected_indices = selector.get_support(indices=True)
    # Get all feature names from TF-IDF
    all_feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
    # Get selected feature names
    feature_names = all_feature_names[selected_indices]
else:
    feature_names = tfidf_vectorizer.get_feature_names_out()

# Different approach based on model type
if 'LogisticRegression' in best_model['model_name'] or 'LinearSVC' in best_model['model_name']:
    # For linear models, extract coefficients
    feature_importances = []

    # Loop through each label's model
    for i, estimator in enumerate(best_model['model'].estimators_):
        # Get coefficients
        if hasattr(estimator, 'coef_'):
            coefs = estimator.coef_[0]
            # Add absolute coefficient values to feature_importances
            if len(feature_importances) == 0:
                feature_importances = np.abs(coefs)
            else:
                feature_importances += np.abs(coefs)

    # Average importance
    feature_importances = feature_importances / len(best_model['model'].estimators_)

elif 'RandomForest' in best_model['model_name']:
    # For RandomForest, extract feature importance
    feature_importances = []

    # Loop through each label's model
    for i, estimator in enumerate(best_model['model'].estimators_):
        # Get feature importance
        if hasattr(estimator, 'feature_importances_'):
            # Add importance values to feature_importances
            if len(feature_importances) == 0:
                feature_importances = estimator.feature_importances_
            else:
                feature_importances += estimator.feature_importances_

    # Average importance
    feature_importances = feature_importances / len(best_model['model'].estimators_)

# Create a DataFrame with feature names and importances
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Sort by importance
feature_importance_df = feature_importance_df.sort_values('Importance', ascending=False)

# Display top 20 important features
print("\nTop 20 important features:")
print(feature_importance_df.head(20))

# Visualize top features
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(20))
plt.title('Top 20 Important Features')
plt.tight_layout()
plt.show()

importance_tracker.stop()
importance_emissions = importance_tracker.final_emissions
print(f"Carbon emissions for feature importance analysis: {importance_emissions:.6f} kg CO2eq")


In [ ]:
# Save the model and all necessary components
import os

# Remove existing lock file if present
if os.path.exists('/tmp/.codecarbon.lock'):
    os.remove('/tmp/.codecarbon.lock')

save_tracker = EmissionsTracker(allow_multiple_runs=True)
save_tracker.start()

print("\nSaving the final model...")

# Define the model data to save
model_data = {
    'model': best_model['model'],
    'vectorizer': tfidf_vectorizer,
    'selector': selector if hasattr(selector, 'get_support') else None,
    'label_columns': label_columns,
    'preprocess_function': preprocess_text,
    'metrics': {
        'sample_accuracy': best_model['sample_accuracy'],
        'overall_accuracy': best_model['overall_accuracy'],
        'f1_macro': best_model['f1_macro'],
        'hamming_loss': best_model['hamming_loss']
    }
}

# Save the model data
joblib.dump(model_data, 'skin_condition_classifier.pkl')
print("Model saved to 'skin_condition_classifier.pkl'")

# Create a prediction function for new descriptions
def predict_skin_conditions(text, model_data):
    """Predict skin conditions for a new product description"""
    # Preprocess text
    processed = model_data['preprocess_function'](text)

    # Transform to TF-IDF
    X = model_data['vectorizer'].transform([processed])

    # Apply feature selection if it exists
    if model_data['selector'] is not None:
        X = model_data['selector'].transform(X)

    # Start emission tracking for inference
    inference_tracker = EmissionsTracker(allow_multiple_runs=True)
    inference_tracker.start()

    # Make prediction
    prediction = model_data['model'].predict(X)[0]

    # Stop emission tracking and capture the return value
    emissions = inference_tracker.stop()

    # Create result dictionary with predictions and metadata
    result = {
        'predictions': {label: pred for label, pred in zip(model_data['label_columns'], prediction)},
        'positive_labels': [label for label, pred in zip(model_data['label_columns'], prediction) if pred == 1],
        'emissions': emissions
    }

    return result

# Test the prediction function
test_text = "This moisturizing cream is perfect for dry, sensitive skin and helps reduce redness and inflammation. Contains hyaluronic acid and is suitable for daily use."
print("\nTesting prediction function with sample text:")
print(f"Sample text: {test_text}")

# Load the saved model to ensure it works
loaded_model_data = joblib.load('skin_condition_classifier.pkl')
prediction_result = predict_skin_conditions(test_text, loaded_model_data)

print("\nPredicted skin conditions/attributes:")
for label in prediction_result['positive_labels']:
    print(f"- {label}")
print(f"Inference carbon emissions: {prediction_result['emissions']:.6f} kg CO2eq")

# Stop tracking emissions for saving model
save_emissions = save_tracker.stop()
print(f"Carbon emissions for saving model: {save_emissions:.6f} kg CO2eq")


In [ ]:
# Step 9: Carbon Emissions Summary

# Remove existing lock file if present
import os
if os.path.exists('/tmp/.codecarbon.lock'):
    os.remove('/tmp/.codecarbon.lock')

# Create a new tracker for this step
from codecarbon import EmissionsTracker
summary_tracker = EmissionsTracker(allow_multiple_runs=True)
summary_tracker.start()

# Stop the main tracker and capture the emissions
total_emissions = tracker.stop()

print("\nCarbon Emissions Summary:")
print("-" * 40)

# Define emission variables with correct values from previous steps
# (These should be captured in each previous step)
try:
    # Print emissions from each step - modify these variable names to match what you captured
    print(f"Preprocessing: {preprocess_emissions:.6f} kg CO2eq")
    print(f"Feature Engineering: {feature_emissions:.6f} kg CO2eq")

    # Sum model training emissions
    training_emissions = sum(result['emissions'] for result in results)
    print(f"Model Training: {training_emissions:.6f} kg CO2eq")

    print(f"Optimization: {opt_emissions:.6f} kg CO2eq")
    print(f"Feature Importance Analysis: {importance_emissions:.6f} kg CO2eq")
    print(f"Model Saving: {save_emissions:.6f} kg CO2eq")
    print(f"Single Inference: {prediction_result['emissions']:.6f} kg CO2eq")
except (NameError, KeyError) as e:
    print(f"Note: Some emission variables weren't available: {e}")
    print("Make sure to capture emissions by saving the return value of stop() method in each step")
    print("Example: emissions = tracker.stop()")

print("-" * 40)
print(f"Total Emissions: {total_emissions:.6f} kg CO2eq")

# Create a visualization of emissions if data is available
try:
    emissions_data = {
        'Preprocessing': preprocess_emissions,
        'Feature Engineering': feature_emissions,
        'Model Training': training_emissions,
        'Optimization': opt_emissions,
        'Feature Importance': importance_emissions,
        'Model Saving': save_emissions
    }

    plt.figure(figsize=(10, 6))
    plt.pie(list(emissions_data.values()), labels=list(emissions_data.keys()),
            autopct='%1.1f%%', startangle=90)
    plt.axis('equal')
    plt.title('Carbon Emissions by Process Step')
    plt.show()
except (NameError, KeyError):
    # If detailed emissions aren't available, create an example visualization
    print("\nCreating example visualization instead of actual emissions breakdown...")

    # Example data - replace with actual values when available
    example_labels = ['Data Processing', 'Feature Engineering', 'Model Training',
                     'Model Optimization', 'Inference']
    example_sizes = [15, 25, 40, 15, 5]  # Example percentages

    plt.figure(figsize=(10, 6))
    plt.pie(example_sizes, labels=example_labels, autopct='%1.1f%%', startangle=90)
    plt.axis('equal')
    plt.title('Example ML Workflow Emissions Distribution')
    plt.show()

# Stop the summary tracker
summary_emissions = summary_tracker.stop()
print(f"\nCarbon emissions for generating this summary: {summary_emissions:.6f} kg CO2eq")
